# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and performing basic analysis on the FAIR^2 dataset using the `mlcroissant` library. You will learn how to interact with the Croissant data model, load record sets, and analyze or visualize records.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://github.com/mlcommons/croissant) accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (not as a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

**All elements are referenced by their `@id` as per Croissant best practice.**

In [ ]:
# Print all record sets and their structure (by @id)
print("Available record sets:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    # List fields for this record set
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    + Field @id: {field.id}, name: {field.name}, type: {field.data_type}")
    # List columns for this record set (if any)
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    + Column @id: {column.id}, name: {column.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we select a record set (choose by `@id` for reproducibility), and show how to extract a DataFrame. **Adjust the chosen `@id` fields as needed based on the overview above.**

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print("All record set @ids\n", record_set_ids)

# For this dataset, typically the main record set contains proteins information
# You can inspect available record sets above and choose one by @id, e.g.:
protein_records_set_id = record_set_ids[0] if record_set_ids else None  # Update if other id is primary!
print(f"Selecting record_set '@id' for extraction: {protein_records_set_id}")

# Load records from all record sets into DataFrames
dataframes = {}
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Loaded record set {recset_id} with shape: {dataframes[recset_id].shape}")

# Show columns for main record set
if protein_records_set_id:
    print(f"Columns in '{protein_records_set_id}':")
    print(dataframes[protein_records_set_id].columns.tolist())
    display(dataframes[protein_records_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as:

- Filtering records based on numeric thresholds (e.g. peptide count, molecular weight)
- Normalizing numeric fields
- Grouping/categorizing records

**Reference fields/columns by their `@id`!**

In [ ]:
# Let's inspect available columns and numeric fields for EDA
df = dataframes[protein_records_set_id]
print("Numeric columns detected:")
numeric_candidates = df.select_dtypes(include=['int64','float64']).columns.tolist()
print(numeric_candidates)

# For demonstration, pick the first numeric field (e.g. peptide count, or molecular weight)
numeric_field = None
if 'cr:peptide_count' in df.columns:
    numeric_field = 'cr:peptide_count'
elif 'cr:molecular_weight' in df.columns:
    numeric_field = 'cr:molecular_weight'
elif numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    print("No numeric fields found for EDA!")

print(f"Selected numeric field for analysis: {numeric_field}")

# Filter records above a threshold
threshold = df[numeric_field].mean() if numeric_field else 0
filtered_df = df[df[numeric_field] > threshold] if numeric_field else df.copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows")

# Normalize the numeric field
if numeric_field:
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )

    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field (e.g. protein type or sample)
group_field = None
for col in ['cr:sample_id','cr:protein_type','cr:accession','cr:sample','cr:group']:
    if col in df.columns:
        group_field = col
        break

print(f"Grouping field: {group_field}")
if group_field and numeric_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped by {group_field} and mean({numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the data using pandas and matplotlib. 

In this section, we plot a histogram of the selected numeric field, and (if grouping field found) a grouped bar chart.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the selected numeric field
if numeric_field:
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Plot average numeric field per group (if available)
if group_field and numeric_field:
    top_n = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.figure(figsize=(10,5))
    plt.bar(top_n[group_field].astype(str), top_n[numeric_field], color="skyblue")
    plt.title(f"Top 10 {group_field}: average {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel(numeric_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded and explored FAIR^2 dataset metadata using `mlcroissant`
- Listed record sets and fields by their `@id`
- Extracted a main data table and performed basic EDA and visualizations

Explore the rest of the fields and record sets using their `@id` values, and customize this notebook for your own secondary analyses!

_To learn more, see [mlcroissant documentation](https://mlcommons.github.io/croissant/)_